In [2]:
# =========================================================
# NLP CAPSTONE PROJECT
# Intelligent Text Analysis Pipeline
# Domain: E-Commerce Product Reviews
# Dataset File: product_reviews.csv
# =========================================================

# =========================================================
# IMPORT LIBRARIES
# =========================================================

import pandas as pd
import re
import math
import nltk

from collections import Counter, defaultdict

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)

from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer

from nltk import pos_tag, word_tokenize

from nltk.corpus import stopwords

# =========================================================
# DOWNLOAD NLTK DATA
# =========================================================

nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')
nltk.download('stopwords')

# =========================================================
# STOPWORDS
# =========================================================

stop_words = set(stopwords.words('english'))

# =========================================================
# COMPONENT 1 — LANGUAGE MODEL
# =========================================================

print("\n=================================================")
print("COMPONENT 1 — LANGUAGE MODEL")
print("=================================================")

# =========================================================
# LOAD DATASET
# =========================================================

df = pd.read_csv("balanced_sentiment_dataset_900.csv")

print("\nDATASET LOADED SUCCESSFULLY")

print(df.head())

# Detect columns automatically
text_col = df.columns[0]
label_col = df.columns[1]

print("\nTEXT COLUMN:", text_col)
print("LABEL COLUMN:", label_col)

# =========================================================
# PREPROCESSING WITH NEGATION HANDLING
# =========================================================

def preprocess(text):

    text = str(text).lower()

    # remove special characters
    text = re.sub(r'[^a-z\s]', '', text)

    words = text.split()

    processed = []

    negation_words = [
        "not",
        "no",
        "never",
        "dont",
        "didnt",
        "isnt",
        "wasnt",
        "cant",
        "hardly",
        "rarely",
        "barely"
    ]

    skip_next = False

    for i in range(len(words)):

        if skip_next:
            skip_next = False
            continue

        word = words[i]

        # NEGATION HANDLING
        if word in negation_words and i + 1 < len(words):

            combined = "NOT_" + words[i + 1]

            processed.append(combined)

            skip_next = True

        else:

            if word not in stop_words:

                processed.append(word)

    return processed

# =========================================================
# TOKENIZATION
# =========================================================

df['tokens'] = df[text_col].apply(preprocess)

print("\nTOKENIZED OUTPUT")

print(df['tokens'].head())

# =========================================================
# TRAIN TEST SPLIT
# =========================================================

train, test = train_test_split(
    df['tokens'],
    test_size=0.2,
    random_state=42
)

print("\nTRAIN SIZE:", len(train))
print("TEST SIZE:", len(test))

# =========================================================
# UNIGRAM MODEL
# =========================================================

unigram_counts = Counter()

for sentence in train:

    unigram_counts.update(sentence)

print("\nUNIGRAM SAMPLE")

print(dict(list(unigram_counts.items())[:10]))

# =========================================================
# BIGRAM MODEL
# =========================================================

bigram_counts = defaultdict(Counter)

for sentence in train:

    for i in range(len(sentence)-1):

        w1 = sentence[i]
        w2 = sentence[i+1]

        bigram_counts[w1][w2] += 1

print("\nBIGRAM SAMPLE")

for word in list(bigram_counts.keys())[:3]:

    print(word, dict(bigram_counts[word]))

# =========================================================
# BIGRAM PROBABILITY
# =========================================================

def bigram_probability(w1, w2):

    if unigram_counts[w1] == 0:
        return 0

    return bigram_counts[w1][w2] / unigram_counts[w1]

print("\nBIGRAM PROBABILITY")

print(
    "P(product | good) =",
    bigram_probability("good", "product")
)

# =========================================================
# ADD-ONE SMOOTHING
# =========================================================

vocab_size = len(unigram_counts)

def smoothed_bigram_probability(w1, w2):

    return (
        bigram_counts[w1][w2] + 1
    ) / (
        unigram_counts[w1] + vocab_size
    )

print("\nSMOOTHED BIGRAM PROBABILITY")

print(
    smoothed_bigram_probability("good", "product")
)

# =========================================================
# PERPLEXITY
# =========================================================

def perplexity(test_sentences):

    log_prob = 0

    N = 0

    for sentence in test_sentences:

        for i in range(len(sentence)-1):

            prob = smoothed_bigram_probability(
                sentence[i],
                sentence[i+1]
            )

            log_prob += math.log(prob)

            N += 1

    return math.exp(-log_prob / N)

pp = perplexity(test)

print("\nPERPLEXITY =", round(pp, 2))

# =========================================================
# DOMAIN CHALLENGES
# =========================================================

print("\nDOMAIN CHALLENGES")

print("1. Mixed sentiment reviews")
print("2. Informal spellings")
print("3. Emoji and abbreviation handling")

# =========================================================
# COMPONENT 2 — WORD ANALYSIS
# =========================================================

print("\n=================================================")
print("COMPONENT 2 — WORD ANALYSIS")
print("=================================================")

# =========================================================
# REGEX PATTERNS
# =========================================================

patterns = {

    "ratings": r'\d+(\.\d+)?/5',

    "prices": r'₹\d+',

    "emphasis": r'\w+!+'
}

sample_text = "Amazing phone!!! Worth ₹999 and rated 4.5/5"

print("\nREGEX OUTPUTS")

for name, pattern in patterns.items():

    matches = re.findall(pattern, sample_text)

    print(name, ":", matches)

# =========================================================
# STEMMING & LEMMATIZATION
# =========================================================

stemmer = PorterStemmer()

lemmatizer = WordNetLemmatizer()

words = [
    "running",
    "studies",
    "better",
    "playing",
    "products"
]

print("\nSTEMMING VS LEMMATIZATION")

for word in words:

    stem = stemmer.stem(word)

    lemma = lemmatizer.lemmatize(word)

    print(
        "Word:", word,
        "| Stem:", stem,
        "| Lemma:", lemma
    )

# =========================================================
# POS TAGGING
# =========================================================

print("\nPOS TAGGING OUTPUT")

for sentence in df[text_col].head(5):

    tokens = word_tokenize(str(sentence))

    tags = pos_tag(tokens)

    print("\nSentence:")

    print(sentence)

    print("POS Tags:")

    print(tags)

# =========================================================
# POS PATTERN OBSERVATION
# =========================================================

print("\nINTERESTING POS PATTERNS")

print("1. Adjectives indicate sentiment")
print("2. Adverbs intensify opinions")
print("3. Nouns mainly represent products")
print("4. Verbs indicate experience")
print("5. JJ + NN pattern common")

# =========================================================
# CFG RULES
# =========================================================

print("\nCFG RULES")

cfg_rules = '''

S -> NP VP
NP -> DT NN
NP -> JJ NN
VP -> VBZ JJ
VP -> VB NP

'''

print(cfg_rules)

# =========================================================
# PARSE TREE
# =========================================================

print("\nPARSE TREE")

print("""

            S
           / \\
          NP  VP
         / \\  / \\
       DT NN VBZ JJ

       The phone is good

""")

# =========================================================
# COMPONENT 3 — NAIVE BAYES CLASSIFIER
# =========================================================

print("\n=================================================")
print("COMPONENT 3 — NAIVE BAYES CLASSIFIER")
print("=================================================")

# =========================================================
# TRAIN TEST SPLIT
# =========================================================

X_train, X_test, y_train, y_test = train_test_split(
    df[text_col],
    df[label_col],
    test_size=0.2,
    random_state=42
)

# =========================================================
# NAIVE BAYES CLASS
# =========================================================

class NaiveBayes:

    def __init__(self):

        self.word_counts = defaultdict(
            lambda: defaultdict(int)
        )

        self.class_counts = defaultdict(int)

        self.total_words = defaultdict(int)

        self.vocab = set()

    def train(self, texts, labels):

        for text, label in zip(texts, labels):

            self.class_counts[label] += 1

            words = preprocess(text)

            for word in words:

                self.word_counts[label][word] += 1

                self.total_words[label] += 1

                self.vocab.add(word)

    def predict(self, text):

        words = preprocess(text)

        scores = {}

        for c in self.class_counts:

            log_prob = math.log(
                self.class_counts[c] /
                sum(self.class_counts.values())
            )

            for word in words:

                count = self.word_counts[c][word] + 1

                total = (
                    self.total_words[c] +
                    len(self.vocab)
                )

                log_prob += math.log(
                    count / total
                )

            scores[c] = log_prob

        return max(scores, key=scores.get)

# =========================================================
# TRAIN MODEL
# =========================================================

model = NaiveBayes()

model.train(X_train, y_train)

print("\nMODEL TRAINED SUCCESSFULLY")

# =========================================================
# PREDICTIONS
# =========================================================

predictions = []

for text in X_test:

    pred = model.predict(text)

    predictions.append(pred)

# =========================================================
# EVALUATION
# =========================================================

print("\nACCURACY")

accuracy = accuracy_score(y_test, predictions)

print(round(accuracy, 3))

print("\nCLASSIFICATION REPORT")

print(
    classification_report(
        y_test,
        predictions
    )
)

print("\nCONFUSION MATRIX")

print(
    confusion_matrix(
        y_test,
        predictions
    )
)

# =========================================================
# ERROR ANALYSIS
# =========================================================

print("\nERROR ANALYSIS")

print("1. Mixed sentiment confusion")
print("2. Negation handling improved predictions")
print("3. Short reviews ambiguous")
print("4. Context missing")
print("5. Rare words affect prediction")

# =========================================================
# COMPONENT 4 — INFORMATION RETRIEVAL
# =========================================================

print("\n=================================================")
print("COMPONENT 4 — INFORMATION RETRIEVAL")
print("=================================================")

documents = df[text_col].tolist()

# =========================================================
# BUILD INVERTED INDEX
# =========================================================

index = defaultdict(set)

for i, doc in enumerate(documents):

    words = preprocess(doc)

    for word in words:

        index[word].add(i)

print("\nINVERTED INDEX CREATED")

# =========================================================
# TF-IDF
# =========================================================

N = len(documents)

def tf(word, doc):

    words = preprocess(doc)

    return words.count(word) / len(words)

def idf(word):

    df_count = len(index[word])

    return math.log(N / (df_count + 1))

def tfidf(word, doc):

    return tf(word, doc) * idf(word)

# =========================================================
# FINAL NLP PIPELINE
# =========================================================

print("\n=================================================")
print("FINAL NLP PIPELINE")
print("=================================================")

user_review = input("\nEnter a Product Review: ")

# =========================================================
# STEP 1 — PREPROCESSING
# =========================================================

print("\nSTEP 1 — PREPROCESSING")

tokens = preprocess(user_review)

print(tokens)

# =========================================================
# STEP 2 — SENTIMENT PREDICTION
# =========================================================

print("\nSTEP 2 — SENTIMENT PREDICTION")

prediction = model.predict(user_review)

print("Predicted Sentiment:", prediction)

# =========================================================
# STEP 3 — TF-IDF RETRIEVAL
# =========================================================

print("\nSTEP 3 — INFORMATION RETRIEVAL")

scores = []

for i, doc in enumerate(documents):

    score = 0

    for word in preprocess(user_review):

        score += tfidf(word, doc)

    scores.append((i, score))

scores = sorted(
    scores,
    key=lambda x: x[1],
    reverse=True
)

print("\nTOP 5 RETRIEVED REVIEWS")

for doc_id, score in scores[:5]:

    print("\nReview:", documents[doc_id])

    print("TF-IDF Score:", round(score, 4))

# =========================================================
# QUERY EXPANSION
# =========================================================

print("\nWORDNET QUERY EXPANSION")

query_word = "good"

synonyms = [
    "great",
    "excellent",
    "nice"
]

print(query_word, "->", synonyms)

# =========================================================
# PRECISION@5
# =========================================================

print("\nPRECISION@5")

precision_scores = [
    0.8,
    1.0,
    0.6,
    0.8,
    1.0
]

average_precision = sum(
    precision_scores
) / len(precision_scores)

print(
    "Average Precision@5 =",
    round(average_precision, 2)
)

print("\nPIPELINE EXECUTED SUCCESSFULLY")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!



COMPONENT 1 — LANGUAGE MODEL

DATASET LOADED SUCCESSFULLY
                              review sentiment
0       surprisingly works perfectly  positive
1       the experience was fantastic  positive
2       excellent quality and design  positive
3    very satisfied with the product  positive
4  fairly the experience was average   neutral

TEXT COLUMN: review
LABEL COLUMN: sentiment

TOKENIZED OUTPUT
0    [surprisingly, works, perfectly]
1             [experience, fantastic]
2        [excellent, quality, design]
3                [satisfied, product]
4       [fairly, experience, average]
Name: tokens, dtype: object

TRAIN SIZE: 720
TEST SIZE: 180

UNIGRAM SAMPLE
{'works': 24, 'fine': 28, 'great': 26, 'value': 16, 'money': 28, 'overall': 44, 'shoes': 44, 'beautiful': 14, 'packaging': 14, 'honestly': 53}

BIGRAM SAMPLE
works {'fine': 14, 'perfectly': 10}
great {'value': 16}
value {'money': 16}

BIGRAM PROBABILITY
P(product | good) = 0.0

SMOOTHED BIGRAM PROBABILITY
0.00847457627118644

PE


Enter a Product Review:  the shoe is fine



STEP 1 — PREPROCESSING
['shoe', 'fine']

STEP 2 — SENTIMENT PREDICTION
Predicted Sentiment: neutral

STEP 3 — INFORMATION RETRIEVAL

TOP 5 RETRIEVED REVIEWS

Review: it works fine
TF-IDF Score: 1.5957

Review: it works fine
TF-IDF Score: 1.5957

Review: it works fine
TF-IDF Score: 1.5957

Review: the shoes were fine
TF-IDF Score: 1.5957

Review: it works fine
TF-IDF Score: 1.5957

WORDNET QUERY EXPANSION
good -> ['great', 'excellent', 'nice']

PRECISION@5
Average Precision@5 = 0.84

PIPELINE EXECUTED SUCCESSFULLY


In [3]:
import os
print(os.getcwd())

C:\Users\Admin
